### “Joint angles were extracted from 2D side-view video using a markerless pose estimation approach. Frames with insufficient landmark visibility were excluded, and short gaps in the angle time series were linearly interpolated to enable robust temporal analysis.”

In [1]:
"""
Stable squat-angle extraction from a SIDE-VIEW smartphone video using MediaPipe Pose.

Outputs:
- CSV with per-frame angles + quality signals
- Optional annotated video (skeleton + angles overlay)

Install:
  pip install opencv-python mediapipe numpy pandas

How to Run:
    It's mandatory to change one variable.
    video_directory - set it to the directory path with the video files
    The results are saved in the same path in a subfolder called 'results'.
    Run the notebook.

Last but not least, a list of references for all "not common" libraries I used:
https://chuoling.github.io/mediapipe/solutions/pose.html
https://github.com/AISoltani/MediaPipe_Pose_Face_Hand_Detection
https://docs.opencv.org/3.4/dd/d43/tutorial_py_video_display.html
https://github.com/casiez/OneEuroFilter/tree/main/python
"""

from pathlib import Path
import math
import os
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import cv2
import numpy as np

# error handling with mediapipe
try:
    import mediapipe as mp
except ImportError as e:
    raise SystemExit("Missing mediapipe. Install with: pip install mediapipe") from e

# -----------------------------
# Geometry helpers (core math)
# goal: compute the angle at point B in triangle A–B–C, in degrees.
# -----------------------------
def angle_abc(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    Returns angle ABC in degrees, where points are 2D np arrays [x, y] in pixels.
    """
    # direction vectors (always from mid to outer landmarks)
    ba = a - b                  # b to a
    bc = c - b                  # b to c
    nba = np.linalg.norm(ba)    # length (magnitude) of vector BA
    nbc = np.linalg.norm(bc)    # length (magnitude) of vector BC
    # checking if values are close to 0, to prevent division by zero
    if nba < 1e-6 or nbc < 1e-6:
        return float("nan")
    cosang = float(np.dot(ba, bc) / (nba * nbc))    # cosine of the angle
    cosang = max(-1.0, min(1.0, cosang))            # make sure -1 <= cosang <= 1, to avoid numerical errors
    return math.degrees(math.acos(cosang))

# TODO not included yet
# def vector_angle_to_vertical(p1: np.ndarray, p2: np.ndarray) -> float:
#     """
#     Angle of vector p1->p2 relative to vertical axis (in degrees).
#     0 = perfectly vertical, 90 = horizontal.
#     Useful for torso/shin lean in side view.
#     """
#     v = p2 - p1                 # direction vector
#     nv = np.linalg.norm(v)      # length (magnitude) of v
#     if nv < 1e-6:               # Checks if the vector is almost zero-length
#         return float("nan")
#     # vertical unit (upwards in image is negative y, but angle magnitude is what matters)
#     vert = np.array([0.0, -1.0], dtype=np.float32)
#     cosang = float(np.dot(v / nv, vert))        # dot product with normalized v and vert
#     cosang = max(-1.0, min(1.0, cosang))        # make sure -1 <= cosang <= 1, to avoid numerical errors
#     return math.degrees(math.acos(cosang))

# -----------------------------
# One Euro Filter (stable smoothing)
# -----------------------------
@dataclass
class OneEuroParams:            # container class for parameters
    min_cutoff: float = 1.0     # lower = smoother
    beta: float = 0.02          # higher = more responsive
    d_cutoff: float = 1.0       # derivative cutoff

class OneEuroFilter:
    """
    One Euro Filter for a single scalar time series.
    Reference: Casiez et al. "The 1€ Filter" (commonly used for pose smoothing).
    """
    def __init__(self, params: OneEuroParams, init_value: Optional[float] = None, init_time: Optional[float] = None):
        self.params = params        # config parameters
        self.x_prev = init_value    # previous filtered value
        self.dx_prev = 0.0          # previous filtered derivative
        self.t_prev = init_time     # previous time

    # computes alpha, the exponential smoothing factor
    @staticmethod
    def _alpha(cutoff: float, dt: float) -> float:
        # alpha = 1 / (1 + tau/dt), tau = 1/(2*pi*cutoff)
        if dt <= 0.0:
            return 1.0
        tau = 1.0 / (2.0 * math.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    # classic exponential smoothing with lowpass filter
    @staticmethod
    def _lowpass(x: float, x_prev: float, alpha: float) -> float:
        return alpha * x + (1.0 - alpha) * x_prev

    def __call__(self, x: float, t: float) -> float:
        if self.t_prev is None or self.x_prev is None or math.isnan(self.x_prev):
            # initialize filter state with the first observation
            self.t_prev = t
            self.x_prev = x
            self.dx_prev = 0.0
            return x

        # compute time step
        dt = t - self.t_prev
        if dt <= 0.0:           # if timestamps are non-increasing, return last filtered value (avoids unstable math)
            return self.x_prev

        # Derivative of the signal
        dx = (x - self.x_prev) / dt
        a_d = self._alpha(self.params.d_cutoff, dt)
        dx_hat = self._lowpass(dx, self.dx_prev, a_d)

        # Adaptive cutoff
        cutoff = self.params.min_cutoff + self.params.beta * abs(dx_hat)
        a = self._alpha(cutoff, dt)
        x_hat = self._lowpass(x, self.x_prev, a)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t
        return x_hat

# -----------------------------
# MediaPipe landmark extraction
# -----------------------------
def mp_landmarks_to_pixels(landmarks, w: int, h: int) -> Dict[int, Tuple[np.ndarray, float]]:
    """
    Returns dict: landmark_index -> (pixel_xy [x,y], visibility)
    """
    out = {}
    for i, lm in enumerate(landmarks.landmark):
        out[i] = (np.array([lm.x * w, lm.y * h], dtype=np.float32), float(getattr(lm, "visibility", 1.0)))
    return out


def pick_visible_side(sample_vis: List[Dict[str, float]]) -> str:
    """
    Decide whether LEFT or RIGHT side is more visible overall.
    Uses median visibility over a short sample.
    """
    left_scores = []
    right_scores = []
    for v in sample_vis:
        left_scores.append(np.mean([v.get("l_hip", 0), v.get("l_knee", 0), v.get("l_ankle", 0), v.get("l_foot", 0)]))
        right_scores.append(np.mean([v.get("r_hip", 0), v.get("r_knee", 0), v.get("r_ankle", 0), v.get("r_foot", 0)]))
    l_med = float(np.median(left_scores)) if left_scores else 0.0
    r_med = float(np.median(right_scores)) if right_scores else 0.0
    return "left" if l_med >= r_med else "right"

# -----------------------------
# Main processing
# -----------------------------
def process_video(
    video_path: str,
    out_csv: str,
    out_video: Optional[str],
    model_complexity: int = 2,
    det_conf: float = 0.7,
    track_conf: float = 0.7,
    visibility_gate: float = 0.6,
    oneeuro: OneEuroParams = OneEuroParams(min_cutoff=1.0, beta=0.02, d_cutoff=1.0),
    sample_frames_for_side: int = 30,
):
    if not os.path.exists(video_path):
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Optional annotated video writer
    writer = None
    if out_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_video, fourcc, float(fps), (w, h))

    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils
    mp_styles = mp.solutions.drawing_styles

    # Indices we care about
    L = mp_pose.PoseLandmark
    IDX = {
        "l_shoulder": int(L.LEFT_SHOULDER),
        "l_hip": int(L.LEFT_HIP),
        "l_knee": int(L.LEFT_KNEE),
        "l_ankle": int(L.LEFT_ANKLE),
        "l_heel": int(L.LEFT_HEEL),
        "l_foot": int(L.LEFT_FOOT_INDEX),

        "r_shoulder": int(L.RIGHT_SHOULDER),
        "r_hip": int(L.RIGHT_HIP),
        "r_knee": int(L.RIGHT_KNEE),
        "r_ankle": int(L.RIGHT_ANKLE),
        "r_heel": int(L.RIGHT_HEEL),
        "r_foot": int(L.RIGHT_FOOT_INDEX),
    }

    # First pass (short) to pick a stable visible side
    sample_vis = []
    sample_imgs = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=model_complexity,
        enable_segmentation=False,
        smooth_landmarks=True,
        min_detection_confidence=det_conf,
        min_tracking_confidence=track_conf,
    ) as pose:
        for i in range(sample_frames_for_side):
            ok, frame = cap.read()
            if not ok:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if res.pose_landmarks:
                pts = mp_landmarks_to_pixels(res.pose_landmarks, w, h)
                v = {k: pts[IDX[k]][1] for k in ["l_hip", "l_knee", "l_ankle", "l_foot", "r_hip", "r_knee", "r_ankle", "r_foot"]}
                sample_vis.append(v)
            sample_imgs.append(frame)

    side = pick_visible_side(sample_vis) if sample_vis else "left"
    # Reset for full processing
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # One Euro filters on angles (more stable than raw per-frame)
    f_knee = OneEuroFilter(oneeuro)
    f_ankle = OneEuroFilter(oneeuro)
    f_hip_proxy = OneEuroFilter(oneeuro)
    f_torso = OneEuroFilter(oneeuro)

    rows = []
    frame_idx = 0

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=model_complexity,
        enable_segmentation=False,
        smooth_landmarks=True,
        min_detection_confidence=det_conf,
        min_tracking_confidence=track_conf,
    ) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            t = frame_idx / float(fps)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)

            knee_ang = ankle_ang = hip_proxy_ang = float("nan")
            gate_ok = False

            if res.pose_landmarks:
                pts = mp_landmarks_to_pixels(res.pose_landmarks, w, h)

                if side == "left":
                    shoulder, v_sh = pts[IDX["l_shoulder"]]
                    hip, v_hip = pts[IDX["l_hip"]]
                    knee, v_knee = pts[IDX["l_knee"]]
                    ankle, v_ank = pts[IDX["l_ankle"]]
                    heel, v_heel = pts[IDX["l_heel"]]
                    foot, v_foot = pts[IDX["l_foot"]]
                else:
                    shoulder, v_sh = pts[IDX["r_shoulder"]]
                    hip, v_hip = pts[IDX["r_hip"]]
                    knee, v_knee = pts[IDX["r_knee"]]
                    ankle, v_ank = pts[IDX["r_ankle"]]
                    heel, v_heel = pts[IDX["r_heel"]]
                    foot, v_foot = pts[IDX["r_foot"]]

                # Visibility gating for stability
                vis_mean = float(np.mean([v_hip, v_knee, v_ank, v_foot]))
                gate_ok = vis_mean >= visibility_gate

                if gate_ok:
                    # Core angles
                    knee_ang = angle_abc(hip, knee, ankle)          # hip-knee-ankle
                    ankle_ang = angle_abc(knee, ankle, foot)        # knee-ankle-foot (dorsi/plantar proxy)
                    hip_proxy_ang = angle_abc(shoulder, hip, knee)  # shoulder-hip-knee (hip+trunk proxy)

                    # One Euro smoothing
                    knee_ang = f_knee(knee_ang, t)
                    ankle_ang = f_ankle(ankle_ang, t)
                    hip_proxy_ang = f_hip_proxy(hip_proxy_ang, t)

                # Draw overlay (optional)
                if writer is not None:
                    mp_draw.draw_landmarks(
                        frame,
                        res.pose_landmarks,
                        mp_pose.POSE_CONNECTIONS,
                        landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
                    )

            rows.append(
                {
                    "frame": frame_idx,
                    "time_s": t,
                    "side_used": side,
                    "gate_ok": int(gate_ok),
                    "ankle_deg": ankle_ang,
                    "knee_deg": knee_ang,
                    "hip_proxy_deg": hip_proxy_ang,
                }
            )

            if writer is not None:
                # Text overlay
                txt = f"side={side} gate={int(gate_ok)}  knee={knee_ang:6.1f}  ankle={ankle_ang:6.1f}  hip*={hip_proxy_ang:6.1f}"
                cv2.putText(frame, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (20, 20, 20), 4, cv2.LINE_AA)
                cv2.putText(frame, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (245, 245, 245), 1, cv2.LINE_AA)
                writer.write(frame)

            frame_idx += 1

    cap.release()
    if writer is not None:
        writer.release()

    df = pd.DataFrame(rows)

    # Optional: fill short gaps (frames failing gate) by interpolation for downstream rep segmentation
    # Keep original columns; add *_interp variants
    for col in ["knee_deg", "ankle_deg", "hip_proxy_deg", "torso_lean_deg"]:
        s = df[col].astype(float)
        df[col + "_interp"] = s.interpolate(limit=10, limit_direction="both")

    df.to_csv(out_csv, index=False, float_format="%.4f")
    print(f"Done.\n- CSV: {out_csv}\n- Annotated video: {out_video if out_video else '(not written)'}")
    print(f"Side chosen: {side} | Frames processed: {len(df)} | FPS: {fps:.2f} | Resolution: {w}x{h}")



'''
The following last section is where the video_directory path need to be set
'''
# video_directory = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\Datenaufnahme"
# output_dir = str(Path(video_directory).parent) + "\\results\\"
video_directory = "C:\\Users\mauri\Desktop\\Uni\WS_25_26\Bachelor\PoseEstimation\\resources\DatenaufnahmeTest"
output_dir = str(Path(video_directory).parent) + "\\resultsTest\\"
model_complexity = 2

for file in os.listdir(video_directory):
    video_path = video_directory + "\\" + file
    out_csv = output_dir + "\\" + Path(video_path).stem + "_" + str(model_complexity) + ".csv"
    out_video = output_dir + "\\" + Path(video_path).stem + "_" + str(model_complexity) + ".mp4"

    params = OneEuroParams(
        min_cutoff=1.0,   # smoother if lower
        beta=0.02,        # responsiveness
        d_cutoff=1.0
    )

    process_video(
        video_path=video_path,
        out_csv=out_csv,
        out_video=out_video,
        model_complexity=model_complexity,   # highest accuracy
        det_conf=0.7,
        track_conf=0.7,
        visibility_gate=0.6,
        oneeuro=params,
    )

KeyboardInterrupt: 